In [ ]:
!pip install -r requirements_vit5.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
import os

In [ ]:

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())


PyTorch: 2.13.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    "VietAI/vit5-base",
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "VietAI/vit5-base"
).to(device)

print("Using:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

Using: cpu


In [ ]:
# -----------------------
# 1. Load data
# -----------------------
dataset = load_dataset("csv", data_files="data\rq1\ec\input\output_phoasr_train.csv")["train"] 

# Small train/validation split
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# -----------------------
# 2. Model
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    "VietAI/vit5-base",
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "VietAI/vit5-base"
).to(device)

# -----------------------
# 3. Tokenization
# -----------------------
def preprocess(example):
    inputs = [
        "correct the transcription: " + text
        for text in example["asr_text"]
    ]

    targets = example["reference_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=1024,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3606 [00:00<?, ? examples/s]

Map:   0%|          | 0/401 [00:00<?, ? examples/s]

In [ ]:
# -----------------------
# 4. Training
# -----------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5-asr-correction",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    predict_with_generate=True,
    #fp16=True,  # remove if using CPU
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./vit5-asr-correction/final")
tokenizer.save_pretrained("./vit5-asr-correction/final")


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


In [ ]:
output_dir = Path("./vit5-asr-correction")

print("Available checkpoints:")

for path in sorted(output_dir.glob("checkpoint-*")):
    print(path)

Available checkpoints:


In [ ]:
CHECKPOINT = "./vit5-asr-correction/final"

tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)

model = T5ForConditionalGeneration.from_pretrained(
    CHECKPOINT
)

model.eval()

print("Loaded model:", model.name_or_path)

In [ ]:
#Save function, here I used my own Drive so you have to choose your own
os.makedirs('/content/drive/MyDrive/vit5-asr-correction', exist_ok=True) 
torch.save(model.state_dict(),'/content/drive/MyDrive/vit5-asr-correction/final.pth')
tokenizer.save_pretrained("/content/drive/MyDrive/vit5-asr-correction/t5_tokenizer")

VALIDATION

In [ ]:
import torch
from jiwer import wer
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
import json

In [ ]:
# ============================================================
# 1. Paths
# ============================================================

MODEL_WEIGHTS = (
    "/content/drive/MyDrive/"
    "vit5-asr-correction/final.pth"
) #Again, you have to find and load where you saved your weight/tokenizer

TOKENIZER_PATH = (
    "/content/drive/MyDrive/"
    "vit5-asr-correction/t5_tokenizer"
)

# ============================================================
# 2. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 3. Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    use_fast=False
)


# ============================================================
# 4. Recreate base ViT5 architecture
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "VietAI/vit5-base"
)


# ============================================================
# 5. Load your fine-tuned weights
# ============================================================

state_dict = torch.load(
    MODEL_WEIGHTS,
    map_location="cpu"
)

model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Fine-tuned ViT5 loaded successfully.")

Device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Fine-tuned ViT5 loaded successfully.


In [ ]:
# ============================================================
# 6. Validation dataset
# ============================================================
dataset = load_dataset("csv", data_files="data\rq1\ec\input\output_phoasr_train.csv")["train"]

# Small train/validation split
dataset = dataset.train_test_split(test_size=0.1, seed=42)

validation_dataset = dataset["test"]

print(
    "Validation examples:",
    len(validation_dataset)
)

In [ ]:
# ============================================================
# JSON output
# ============================================================

JSON_PATH = (
    "/content/drive/MyDrive/"
    "vit5-asr-correction/validation_results.json" #Not included in the repository because this is just validation
)

results = []

# ============================================================
# Evaluate
# ============================================================

asr_wers = []
vit5_wers = []

improved = 0
unchanged = 0
worsened = 0


for i, example in enumerate(validation_dataset):

    asr_text = example["asr_text"]
    reference = example["reference_text"]

    prompt = "correct the transcription: " + asr_text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # WER
    # --------------------------------------------------------

    asr_wer = wer(reference, asr_text)
    vit5_wer = wer(reference, prediction)

    asr_wers.append(asr_wer)
    vit5_wers.append(vit5_wer)
    change_wer = wer(asr_text, prediction)

    # --------------------------------------------------------
    # Result category
    # --------------------------------------------------------

    if vit5_wer < asr_wer:
        result = "improved"
        improved += 1

    elif vit5_wer > asr_wer:
        result = "worsened"
        worsened += 1

    else:
        result = "unchanged"
        unchanged += 1

    # --------------------------------------------------------
    # Save individual result
    # --------------------------------------------------------

    results.append({
    "index": i,
    "asr_text": asr_text,
    "vit5_prediction": prediction,
    "reference_text": reference,

    "asr_wer": asr_wer,
    "vit5_wer": vit5_wer,

    "asr_to_vit5_wer": change_wer,

    "wer_change": vit5_wer - asr_wer,

    "result": result
})

    print(
        f"{i + 1}/{len(validation_dataset)} | "
        f"ASR WER: {asr_wer:.4f} | "
        f"ViT5 WER: {vit5_wer:.4f} | "
        f"{result}"
    )


# ============================================================
# Average / summary
# ============================================================

mean_asr_wer = sum(asr_wers) / len(asr_wers)
mean_vit5_wer = sum(vit5_wers) / len(vit5_wers)

absolute_reduction = mean_asr_wer - mean_vit5_wer

relative_reduction = (
    absolute_reduction / mean_asr_wer
    if mean_asr_wer > 0
    else 0
)

summary = {
    "num_examples": len(validation_dataset),

    "mean_asr_wer": mean_asr_wer,
    "mean_vit5_wer": mean_vit5_wer,

    "absolute_wer_reduction": absolute_reduction,
    "relative_wer_reduction": relative_reduction,

    "improved": improved,
    "unchanged": unchanged,
    "worsened": worsened
}


# ============================================================
# Append summary to JSON
# ============================================================

output = {
    "results": results,
    "summary": summary
}

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Print summary
# ============================================================

print()
print("=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(f"Examples:               {len(validation_dataset)}")
print(f"Mean ASR WER:           {mean_asr_wer:.4f}")
print(f"Mean ViT5 WER:          {mean_vit5_wer:.4f}")
print(f"Absolute WER reduction: {absolute_reduction:.4f}")
print(
    f"Relative WER reduction: "
    f"{relative_reduction * 100:.2f}%"
)

print(f"Improved:               {improved}")
print(f"Unchanged:              {unchanged}")
print(f"Worsened:               {worsened}")

print()
print(f"Saved to: {JSON_PATH}")

1/401 | ASR WER: 0.0925 | ViT5 WER: 0.1013 | worsened
2/401 | ASR WER: 0.2906 | ViT5 WER: 0.1068 | improved
3/401 | ASR WER: 0.1778 | ViT5 WER: 0.1259 | improved
4/401 | ASR WER: 0.5204 | ViT5 WER: 0.1855 | improved
5/401 | ASR WER: 0.3090 | ViT5 WER: 0.1124 | improved
6/401 | ASR WER: 0.1554 | ViT5 WER: 0.0777 | improved
7/401 | ASR WER: 0.1262 | ViT5 WER: 0.0701 | improved
8/401 | ASR WER: 0.3667 | ViT5 WER: 0.0733 | improved
9/401 | ASR WER: 0.2188 | ViT5 WER: 0.1250 | improved
10/401 | ASR WER: 0.3870 | ViT5 WER: 0.2835 | improved
11/401 | ASR WER: 0.0709 | ViT5 WER: 0.0472 | improved
12/401 | ASR WER: 0.1676 | ViT5 WER: 0.0649 | improved
13/401 | ASR WER: 0.1667 | ViT5 WER: 0.0682 | improved
14/401 | ASR WER: 0.2301 | ViT5 WER: 0.1239 | improved
15/401 | ASR WER: 0.2946 | ViT5 WER: 0.1163 | improved
16/401 | ASR WER: 0.3280 | ViT5 WER: 0.0480 | improved
17/401 | ASR WER: 0.4515 | ViT5 WER: 0.1352 | improved
18/401 | ASR WER: 0.0746 | ViT5 WER: 0.0746 | unchanged
19/401 | ASR WER: 

Full Training

In [ ]:
# -----------------------
# 1. Load data
# -----------------------
dataset = load_dataset(
    "csv",
    data_files="data\rq1\ec\input\output_phoasr_train.csv"
)["train"]

print("Training examples:", len(dataset))


# -----------------------
# 2. Model
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    "VietAI/vit5-base",
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "VietAI/vit5-base"
).to(device)


# -----------------------
# 3. Tokenization
# -----------------------
def preprocess(example):
    inputs = [
        "correct the transcription: " + text
        for text in example["asr_text"]
    ]

    targets = example["reference_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=1024,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset.column_names,
)

In [ ]:
# -----------------------
# 4. Training
# -----------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

training_args = Seq2SeqTrainingArguments( #args used in the final report
    output_dir="./vit5-asr-correction",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=False,
    fp16=True,
    seed=42,
    data_seed=42,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,

    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
os.makedirs('/content/drive/MyDrive/vit5-asr-correction', exist_ok=True)
torch.save(model.state_dict(),'/content/drive/MyDrive/vit5-asr-correction/final.pth')
tokenizer.save_pretrained("/content/drive/MyDrive/vit5-asr-correction/final")